# Lab 06-01 · Signs, Power Balance, Energy Budgets and the Limits of Lumped Circuits

Companion to chapter [06-01](../../chapters/06-01-charge-current-voltage-power.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running. Parts 1, 2 and 4 call ngspice if it is installed; without it they print a note and the rest of the notebook still runs.

In [1]:
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None

def ngspice_values(netlist, names):
    '''Run a netlist in batch mode and return the "name = value" lines it prints, as floats.'''
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        if "=" in line:
            key, _, rest = line.partition("=")
            key = key.strip().lower()
            if key in names:
                vals[key] = float(rest.split()[0])
    return vals

print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — Signs in SPICE

Example 06-01.2: a 12 V supply with 2 Ω and a 6 V battery with 3 Ω feed node A, which has a 6 Ω load to ground.

**🔮 Predict:** what sign does ngspice report for `i(V1)`, the 12 V supply's current? For `@V1[p]` and `@V2[p]`, the powers of the two sources?

First, solve it ourselves: KCL at node A gives one equation in one unknown.

In [2]:
v_a = (12 / 2 + 6 / 3) / (1 / 2 + 1 / 3 + 1 / 6)        # KCL at A: sum of (V_k - v_a)/R_k = v_a/R_3
i1, i2, i3 = (12 - v_a) / 2, (6 - v_a) / 3, v_a / 6
absorbed = {                                               # passive sign convention for every element
    "V1": 12 * (-i1), "R1": (12 - v_a) * i1,
    "V2": 6 * (-i2), "R2": (6 - v_a) * i2, "R3": v_a * i3,
}
print(f"v_A = {v_a:.3f} V, i1 = {i1:.3f} A, i2 = {i2:.3f} A, i3 = {i3:.3f} A")
for name, p in absorbed.items():
    print(f"  {name}: absorbs {p:+8.3f} W")
print(f"sum = {sum(absorbed.values()):.2e} W  (Tellegen)")
assert abs(sum(absorbed.values())) < 1e-12

v_A = 8.000 V, i1 = 2.000 A, i2 = -0.667 A, i3 = 1.333 A
  V1: absorbs  -24.000 W
  R1: absorbs   +8.000 W
  V2: absorbs   +4.000 W
  R2: absorbs   +1.333 W
  R3: absorbs  +10.667 W
sum = 1.42e-14 W  (Tellegen)


In [3]:
names = {"va", "iv1", "iv2", "pv1", "pr1", "pv2", "pr2", "pr3", "ptotal", "vcross"}
if HAVE_NGSPICE:
    sim = ngspice_values("power_balance.cir", names)
    for k in ("va", "iv1", "iv2", "pv1", "pv2", "pr1", "pr2", "pr3", "ptotal"):
        print(f"  {k:7} = {sim[k]: .6g}")
    assert abs(sim["iv1"] + 2) < 1e-9 and abs(sim["pv1"] + 24) < 1e-9
else:
    print("ngspice not found; see SETUP.md. Skipping.")

  va      =  8
  iv1     = -2
  iv2     =  0.666667
  pv1     = -24
  pv2     =  4
  pr1     =  8
  pr2     =  1.33333
  pr3     =  10.6667
  ptotal  =  0


SPICE treats sources with the same passive convention as resistors: a source's current is positive when it flows *into* its + terminal. The 12 V supply delivers power, so its current out of the + terminal is reported as **−2 A** and its absorbed power as **−24 W**. The battery shows **+0.667 A** and **+4 W**: it is being charged. The five powers sum to zero, as Tellegen's theorem requires.

## Part 2 — A battery that becomes a load

**🔮 Predict:** for what battery voltage $V_2$ does the battery carry no current? Above that value, does it absorb or deliver power?

In [4]:
v2 = np.linspace(0, 12, 1201)
v_a2 = (12 / 2 + v2 / 3) / (1 / 2 + 1 / 3 + 1 / 6)
i2_sweep = (v2 - v_a2) / 3                                 # current out of the battery's + terminal
p_batt = v2 * (-i2_sweep)                                  # absorbed power, passive convention
cross = v2[np.argmin(np.abs(i2_sweep[1:])) + 1]
print(f"battery current changes sign at V2 = {cross:.2f} V")
if HAVE_NGSPICE:
    print(f"ngspice sweep finds V2 = {ngspice_values('power_balance.cir', names)['vcross']:.4f} V")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(v2, p_batt)
ax.axhline(0, color="0.5", lw=0.8)
ax.axvline(cross, color="0.5", lw=0.8, ls="--")
ax.set_xlabel("battery voltage $V_2$ (V)")
ax.set_ylabel("power absorbed by battery (W)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

battery current changes sign at V2 = 9.00 V
ngspice sweep finds V2 = 9.0000 V


Below 9 V the curve is positive: the supply charges the battery. Above 9 V it is negative: the battery delivers power. (The power is also zero at $V_2$ = 0, trivially, because a 0 V "battery" absorbs nothing whatever its current.) At exactly 9 V no current flows through $R_2$, so node A is just the divider $12 \times 6/(2 + 6)$ of the supply.

## Part 3 — An energy budget

Example 06-01.4's sensor tag: 2 µA asleep, 1.2 mA for 20 ms every second, 8 mA for 3 ms every 10 s, from a 225 mAh cell.

**🔮 Predict:** which shortens battery life more: doubling the sensing time to 40 ms, or sending a radio packet every second instead of every 10 s?

In [5]:
def profile(sense_s=0.020, radio_period=10.0, span=10.0, dt=1e-4):
    t = np.arange(0, span, dt)
    i = np.full_like(t, 2e-6)
    i += np.where((t % 1.0) < sense_s, 1.2e-3, 0.0)
    i += np.where((t % radio_period) < 3e-3, 8e-3, 0.0)
    return t, i

def life_days(sense_s=0.020, radio_period=10.0):
    _, i = profile(sense_s, radio_period, dt=1e-5)
    return 0.225 / i.mean() / 24

t, i = profile()
print(f"average current {i.mean() * 1e6:.1f} uA (formula: 28.4 uA); battery life {life_days():.1f} days")
for label, args in (("baseline (20 ms, 10 s)", (0.020, 10.0)), ("sensing 10 ms", (0.010, 10.0)),
                    ("sensing 40 ms", (0.040, 10.0)), ("radio every 1 s", (0.020, 1.0))):
    print(f"  {label:24} {life_days(*args):7.1f} days")

average current 28.4 uA (formula: 28.4 uA); battery life 330.1 days


  baseline (20 ms, 10 s)     330.1 days


  sensing 10 ms              571.6 days
  sensing 40 ms              178.9 days
  radio every 1 s            187.5 days


Halving the sensing time to 10 ms extends life from 330 to 572 days; doubling it to 40 ms cuts life to 179 days. Sending a packet every second cuts it to 187.5 days: the two changes cost about the same, although the radio's current is almost seven times the sensing current, because the radio's duty cycle is so much smaller. The budget is set by current × duty cycle, not by the largest number on the datasheet.

## Part 4 — When Kirchhoff fails

A 1 V step with rise time $t_r$ drives a 1 ns transmission line (about 14 cm of FR-4 stripline) through a 10 Ω driver into an open load. A lumped wire would deliver the step unchanged. A **bounce diagram** follows the waves: the step launches $V_{\text{in}}Z_0/(R_s + Z_0)$ down the line; each arrival at the open end doubles ($\Gamma_L \approx +1$); each return to the 10 Ω source reflects with $\Gamma_S = (10 - 50)/(10 + 50) = -2/3$.

**🔮 Predict:** for which of 0.1, 2, 6 and 20 ns edges does the far end overshoot 1 V by more than 10%?

In [6]:
def far_end(t, tr, rs=10.0, z0=50.0, td=1e-9, rl=1e6):
    gs, gl = (rs - z0) / (rs + z0), (rl - z0) / (rl + z0)
    out = np.zeros_like(t)
    for k in range(300):
        out += z0 / (rs + z0) * (1 + gl) * (gl * gs) ** k * np.clip((t - (2 * k + 1) * td) / tr, 0, 1)
    return out

t = np.linspace(0, 140e-9, 140001)
peaks = {}
fig, ax = plt.subplots(figsize=(7, 3.6))
for tr in (0.1e-9, 2e-9, 6e-9, 20e-9):
    vl = far_end(t, tr)
    peaks[tr] = vl.max()
    ax.plot(t * 1e9, vl, label=f"{tr * 1e9:g} ns edge")
ax.set_xlim(0, 40)
ax.set_xlabel("time (ns)")
ax.set_ylabel("far-end voltage (V)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
for tr, pk in peaks.items():
    print(f"edge {tr * 1e9:5.1f} ns: peak {pk:.4f} V, overshoot {100 * (pk - 1):5.1f}%")
if HAVE_NGSPICE:
    sim = ngspice_values("tline_edges.cir", {"vmax01", "vmax2", "vmax6", "vmax20"})
    print("ngspice peaks:", {k: round(v, 4) for k, v in sim.items()})

edge   0.1 ns: peak 1.6666 V, overshoot  66.7%
edge   2.0 ns: peak 1.6666 V, overshoot  66.7%
edge   6.0 ns: peak 1.1728 V, overshoot  17.3%
edge  20.0 ns: peak 1.0393 V, overshoot   3.9%


ngspice peaks: {'vmax01': 1.6666, 'vmax2': 1.6664, 'vmax6': 1.1728, 'vmax20': 1.0393}


The 0.1 ns and 2 ns edges both reach 1.667 V: the far end sees the doubled incident wave before the first reflection from the source returns ($3t_d$ = 3 ns), so any edge faster than the 2 ns round trip gives the full overshoot. The 6 ns edge, at the $t_d < t_r/6$ boundary, still overshoots by 17% with this badly matched driver; the 20 ns edge by 3.9%. The bounce diagram and ngspice agree to four digits. Where the overshoot matters, the remedy is not a slower edge but a matched source or termination (Course 24).

## Your turn

1. In Part 1, change $R_2$ to 1 Ω. Does the battery still absorb power? Recompute and check the power balance.
2. In Part 4, set the driver resistance to 50 Ω (matched). What is the far-end peak for the 0.1 ns edge, and why?
3. In Part 3, the cell's usable capacity falls 20% in the cold. How many days does the baseline tag last?

<details><summary>Answers</summary>

1. With $R_2$ = 1 Ω, KCL gives $(12 - v_A)/2 + (6 - v_A)/1 = v_A/6$, so $v_A(1/2 + 1 + 1/6) = 12$ and $v_A$ = 7.2 V. The battery's current into its + terminal is $(7.2 - 6)/1 = 1.2$ A: it absorbs $6 \times 1.2 = 7.2$ W, more than before, because the smaller resistance lets the supply push more current into it. Powers: supply $12 \times (-2.4) = -28.8$ W; $R_1$: $4.8 \times 2.4 = 11.52$ W; battery 7.2 W; $R_2$: $1.2^2 \times 1 = 1.44$ W; load $7.2^2/6 = 8.64$ W; sum $-28.8 + 11.52 + 7.2 + 1.44 + 8.64 = 0$ ✓.
2. With $R_s = Z_0$ the source reflection coefficient is 0, so the wave launched is $1 \times 50/100 = 0.5$ V, it doubles at the open end to exactly 1.0 V, and the reflection is absorbed when it returns to the matched source: one clean step, no overshoot, whatever the edge rate. This "source termination" is how many point-to-point digital links are designed.
3. $0.8 \times 225 = 180$ mAh; $180/0.0284 = 6338$ h $= 264$ days.

</details>